# A6 — EDA, Evaluation Protocol Lock, and Pre-Registration

The gate before Study B. Three deliverables:

1. **EDA** across all three layers, reported separately (CLAUDE.md rule 12).
2. **`reports/evaluation_protocol.md`** — the locked protocol (rule 11).
3. **`reports/analysis_plan_preregistration.md`** — hypotheses fixed before any model runs (rule 13).

Analysis lives in `src/evaluation/eda.py`; metrics and splits that Study B will use are in `src/evaluation/metrics.py` and `src/evaluation/splits.py`.

In [1]:
import os
import sys
from pathlib import Path

# Every src/ path is relative to the project root (data/..., outputs/...).
# Without this chdir a run from notebooks/ writes into notebooks/data/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.evaluation.eda import run, log_stats

In [2]:
results = run()
log_stats(results)

2026-11-23 17:52:36,074 INFO src.ingestion.epss: Fetched EPSS as of 2023-01-01 for 192048 CVEs


2026/11/23 17:52:36 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/spopa/Documents/Internship Project/internship-project/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026-11-23 17:52:37,294 INFO src.tracking: Logged A6 to MLflow (run 49ae364b25c2436ca8b2fedd4442a1cb)



A6 exploratory data analysis
Layers are reported separately (rule 12) - they do not share a label source.

         layer   rows     unit  positive_label  positive_rate
     L1 (real) 430678     CVEs        CISA KEV        0.00416
     L2 (real)   3870   checks n/a (no labels)            NaN
L3 (simulated)  39989 findings formula (60/40)        0.05577

L1 - class imbalance:
  KEV positives:    1,790 of 430,678  (1:240)
  public_exploit:   20.6% of CVEs
  CVSS available:   92.4% of CVEs

EPSS LEAKAGE (drives the protocol's point-in-time rule):
                group      n  median_epss_current  median_epss_asof_split
         never in KEV 202614              0.01419                 0.01055
KEV before 2023-01-01    864              0.61775                 0.08631
 KEV after 2023-01-01    174              0.70279                 0.06047

  ratio with EPSS today:        49.5x
  ratio with EPSS at split:     5.7x
  -> today's EPSS encodes post-split outcomes; use fetch_epss_asof().

L2 - s

## The headline finding: EPSS leaks under a temporal split

This is what forces the point-in-time rule in the locked protocol.

Take only CVEs published **before** the split date — the rows that would be training data — and group them by *when* they entered KEV. If a CVE whose exploitation only became public **after** the split already scores high on **today's** EPSS, then that feature carries the outcome we are trying to predict.

In [3]:
results["leakage"]

,group,n,median_epss_current,median_epss_asof_split
0,never in KEV,202614,0.01419,0.01055
1,KEV before 2023-01-01,864,0.61775,0.08631
2,KEV after 2023-01-01,174,0.70279,0.06047


Median EPSS **today** for CVEs exploited only after the split is 45.7× the never-exploited baseline — and indistinguishable from those exploited *before* it. Today's EPSS is close to an oracle for future KEV membership.

Swapping in **point-in-time** EPSS (`src.ingestion.epss.fetch_epss_asof`) drops that ratio to **5.0×**. The residual is legitimate predictive signal, which is exactly what EPSS is designed to provide.

**Binding rule:** `epss_scores-current.csv.gz` may not be a feature in any temporally split experiment. It stays permitted for descriptive statistics and as a B1 baseline ranker (where no training happens).

## Splits and metrics behave as specified

The guards refuse to return a broken split, and the metric set demotes ROC-AUC for a reason worth seeing directly.

In [4]:
import numpy as np
import pandas as pd

from src.evaluation.metrics import evaluate
from src.evaluation.splits import assert_no_temporal_leakage, grouped_split, temporal_split

train, test = temporal_split(results["l1"], "2023-01-01")
assert_no_temporal_leakage(train, test)
print(f"L1 temporal split: train={len(train):,}  test={len(test):,}")

l3_train, l3_test = grouped_split(results["l3"])
print(f"L3 grouped split : train={len(l3_train):,}  test={len(l3_test):,}")

# Why ROC-AUC is secondary: a random ranker still looks respectable on it.
rng = np.random.default_rng(0)
y = np.zeros(10_000)
y[:50] = 1
rng.shuffle(y)
random_scores = evaluate(y, rng.random(10_000))
pd.Series(
    {k: random_scores[k] for k in ["roc_auc", "pr_auc", "precision_at_10", "base_rate"]}
).round(4)

2026-11-23 17:52:37,762 INFO src.evaluation.splits: Temporal split at 2023-01-01: train=203652 (to 2022-12-31), test=227026 (from 2023-01-01)


2026-11-23 17:52:37,781 INFO src.evaluation.splits: Grouped split on asset_id: train=30277 rows / 3750 groups, test=9712 rows / 1249 groups


L1 temporal split: train=203,652  test=227,026
L3 grouped split : train=30,277  test=9,712


roc_auc            0.5025
pr_auc             0.0051
precision_at_10    0.0000
base_rate          0.0050
dtype: float64

## What is now locked

Read `reports/evaluation_protocol.md` and `reports/analysis_plan_preregistration.md` in full before starting B1.

- **Primary metrics:** PR-AUC, Precision@k (k = 10/50/100). ROC-AUC reported, never used to select.
- **Splits:** L1 temporal (never shuffled), L3 grouped-by-asset.
- **EPSS:** point-in-time only in trained experiments.
- **Seeds:** ≥10, mean ± std. Friedman + Nemenyi across families, Wilcoxon pairwise.
- **Baselines first:** no learned model is reported before B1. A model that cannot beat CVSS ordering is reported as a negative result, not buried.
- **L1/L2 and L3 always reported in separate tables.**

**Still open (needs supervisor sign-off):** the L3 label formula puts L1 and L2 on incommensurate scales, so 99.4% of vulnerability findings label `Low`. Until that is resolved, L3 results speak only to *misconfiguration* prioritisation. See the A5 entry in the CLAUDE.md Deviations Log.